# **Convert EmoNet to ONNX and Export**

In [ ]:
!pip install torch torchvision onnx onnxruntime

In [ ]:
!git clone https://github.com/face-analysis/emonet.git
%cd emonet
!ls

In [ ]:
%cd /content/emonet
!ls

In [ ]:
%cd /content/emonet
!pwd
!ls


In [ ]:
import sys
sys.path.append("/content/emonet")

In [ ]:
from emonet.models.emonet import EmoNet
print("EmoNet imported successfully")


In [ ]:
import sys
sys.path.append("/content/emonet")

import torch
import torch.nn as nn
from emonet.models.emonet import EmoNet

class EmoNetVAWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = EmoNet(n_expression=8)

    def forward(self, x):
        out = self.model(x)
        v = out["valence"]
        a = out["arousal"]
        return torch.stack([v.view(-1), a.view(-1)], dim=1)

device = "cpu"
model = EmoNetVAWrapper().to(device)

state = torch.load("pretrained/emonet_8.pth", map_location=device)
model.model.load_state_dict(state)
model.eval()

dummy = torch.randn(1, 3, 256, 256)

onnx_path = "/content/emonet_va.onnx"

with torch.inference_mode():
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["va"],
        dynamic_axes={"input": {0: "batch"}, "va": {0: "batch"}},
        opset_version=18,     # IMPORTANT
        do_constant_folding=True
    )

print("export completed:", onnx_path)


In [ ]:
from google.colab import files
files.download("/content/emonet_va.onnx")